# LFM3-Omni-2B — text, image, video, and audio smoke on a free T4

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/donvito/notebooks/blob/melvin/lfm3-colab-notebooks-103d/colab/LFM3-Omni-2B.ipynb)

[`LiquidAI/LFM3-Omni-2B`](https://huggingface.co/LiquidAI/LFM3-Omni-2B) is Liquid AI's first **omni** LFM3 (~2.17B, `lfm3_omni`): text, images, video, and 16 kHz audio in; text and 24 kHz speech out. One on-device model instead of separate ASR, VLM, LLM, and TTS. It is **not** a decision model — see [`d1-Omni-2B-inference-decisions.ipynb`](d1-Omni-2B-inference-decisions.ipynb) for calibrated yes/no over the same backbone.

Card install: `transformers>=5.20.2`, `torchvision`, `pillow`, `librosa`, `soundfile`, `trust_remote_code`. The card also lists `torchcodec`; Colab often cannot load it, so this notebook writes local files and uses `av`. Native support is still landing in [`huggingface/transformers#50036`](https://github.com/huggingface/transformers/pull/50036).

**Licence:** [LFM Open License v1.0](https://huggingface.co/LiquidAI/LFM3-Omni-2B/blob/main/LICENSE) (`license: other`, `lfm1.0`).

**Runtime:** *Runtime > Change runtime type > T4 GPU*. **float16** + **SDPA**. About 4.3 GB of weights plus LiquidCodec. Video is capped at **8 frames / 2 fps**. Demos ask for **text** (`generation_mode="text"`) so the T4 is not also decoding speech.

**Links:** [model card](https://huggingface.co/LiquidAI/LFM3-Omni-2B) · [blog](https://www.liquid.ai/blog/lfm3) · [announcement](https://x.com/liquidai/status/2115128329003303107) · [collection](https://huggingface.co/collections/LiquidAI/lfm3) · [GGUF](https://huggingface.co/LiquidAI/LFM3-Omni-2B-GGUF)


In [ ]:
%pip install -q -U "transformers>=5.20.2" accelerate
%pip install -q pillow torchvision av librosa soundfile

# Card pin: transformers>=5.20.2 plus torchvision, torchcodec, accelerate,
# pillow, librosa, soundfile. Skip torchcodec (often broken on Colab) and
# do not reinstall torch. av writes the short video.


> **Restart the runtime now:** *Runtime > Restart session*, then continue from the next cell.


## 1. Check the T4 and pick a dtype


In [ ]:
import os
import torch
import transformers
from packaging.version import Version

os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"

assert Version(transformers.__version__) >= Version("5.20.2"), (
    f"transformers {transformers.__version__} is loaded; this notebook needs 5.20.2+. "
    "Run the install cell, then Runtime > Restart session."
)

if not torch.cuda.is_available():
    raise SystemExit("Select Runtime > Change runtime type > T4 GPU, then rerun from the top.")

# T4 (sm_75) has no FlashAttention-2. Keep PyTorch SDPA on mem-efficient and math kernels.
torch.backends.cuda.enable_flash_sdp(False)
torch.backends.cuda.enable_mem_efficient_sdp(True)
torch.backends.cuda.enable_math_sdp(True)

# The cards default to bfloat16 on GPU. A T4 has no native BF16, so use float16.
major, minor = torch.cuda.get_device_capability()
DTYPE = torch.float16
DEVICE = "cuda"
props = torch.cuda.get_device_properties(0)
print("transformers", transformers.__version__)
print("torch", torch.__version__)
print("GPU:", torch.cuda.get_device_name(), f"sm_{major}{minor}", "| dtype:", DTYPE)
print(f"VRAM total: {props.total_memory / 1024**3:.1f} GB")
print("FlashAttention-2: off (sdpa / eager only)")
if major < 8:
    print("No native bfloat16 on this GPU; loading float16.")
if major >= 8:
    print("This GPU can run bfloat16; this notebook still uses float16 so a T4 recipe stays consistent.")


## 2. Load LFM3-Omni-2B

Card snippet: `AutoModelForMultimodalLM` + `AutoProcessor`, then `processor.audio_tokenizer` on the GPU. Omni is a **non-reasoning** model: every answer opens with an empty `<think></think>`.


In [ ]:
from transformers import AutoModelForMultimodalLM, AutoProcessor

MODEL_ID = "LiquidAI/LFM3-Omni-2B"

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()


def load_omni(attn):
    return AutoModelForMultimodalLM.from_pretrained(
        MODEL_ID,
        dtype=DTYPE,
        device_map=DEVICE,
        attn_implementation=attn,
        trust_remote_code=True,
    ).eval()


ATTN = None
model = None
last_error = None
for attn in ("sdpa", "eager"):
    try:
        model = load_omni(attn)
        ATTN = attn
        break
    except Exception as exc:
        last_error = exc
        print(f"{attn} float16 load failed:", exc)
        del model
        model = None
        torch.cuda.empty_cache()

if model is None:
    raise RuntimeError("could not load LFM3-Omni-2B") from last_error

processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)
codec = processor.audio_tokenizer.to(device=DEVICE).eval()

print("attention:", ATTN, "| dtype:", getattr(model, "dtype", DTYPE))
print(f"parameters: {sum(p.numel() for p in model.parameters()) / 1e9:.2f}B")
print(
    f"VRAM allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB | "
    f"peak: {torch.cuda.max_memory_allocated() / 1024**3:.2f} GB"
)


def generate_text(conversation, max_new_tokens=128, **template_kwargs):
    inputs = processor.apply_chat_template(
        conversation,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
        **template_kwargs,
    ).to(DEVICE)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            generation_mode="text",
        )
    text = processor.decode(
        outputs.sequences[0, inputs["input_ids"].shape[1] : outputs.sequence_lengths[0]],
        skip_special_tokens=True,
    )
    print(
        f"VRAM allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB | "
        f"peak: {torch.cuda.max_memory_allocated() / 1024**3:.2f} GB"
    )
    return text


## 3. Demo 1 — image question (text out)

Same public COCO cats photo as the VL and d1 notebooks.


In [ ]:
from pathlib import Path
from urllib.request import urlretrieve

from IPython.display import display
from transformers.image_utils import load_image

IMAGE_URL = "http://images.cocodataset.org/val2017/000000039769.jpg"
IMAGE_PATH = Path("lfm3_omni_cats.jpg")
urlretrieve(IMAGE_URL, IMAGE_PATH)
image = load_image(str(IMAGE_PATH))
display(image.resize((320, 240)))

print(generate_text([
    {
        "role": "system",
        "content": "Answer questions about the provided image. Base every answer only on what can be seen.",
    },
    {
        "role": "user",
        "content": [
            {"type": "image", "image": str(IMAGE_PATH)},
            {"type": "text", "text": "What is in this picture? How many animals?"},
        ],
    },
]))


## 4. Demo 2 — ASR (text out)

Public LibriSpeech dummy clip, same fixture as the Audio notebook.


In [ ]:
import librosa
import soundfile as sf
from IPython.display import Audio, display

ASR_URL = "https://huggingface.co/datasets/Narsil/asr_dummy/resolve/main/1.flac"
ASR_PATH = Path("lfm3_omni_asr.wav")
try:
    src = Path("lfm3_omni_asr.flac")
    urlretrieve(ASR_URL, src)
    wav, _ = librosa.load(src, sr=16_000, mono=True)
    sf.write(ASR_PATH, wav, 16_000)
except Exception as exc:
    print("download failed, synthesizing:", exc)
    import shutil
    import subprocess

    if not shutil.which("espeak-ng"):
        subprocess.run(["apt-get", "-qq", "update"], check=True)
        subprocess.run(["apt-get", "-qq", "install", "-y", "espeak-ng"], check=True)
    subprocess.run(
        ["espeak-ng", "-s", "140", "-v", "en-us", "-w", str(ASR_PATH),
         "The birch canoe slid on the smooth planks."],
        check=True,
    )

wav, sr = sf.read(ASR_PATH)
display(Audio(wav, rate=sr))
print(generate_text([
    {"role": "system", "content": "Perform ASR in English."},
    {"role": "user", "content": [{"type": "audio", "audio": str(ASR_PATH)}]},
]))


## 5. Demo 3 — short video (frames only)

8 frames at 2 fps. Soundtrack interleaving (`load_audio_from_video` / `use_audio_in_video`) is left off so the T4 only pays for frames. Raise `max_frames` on a larger GPU.


In [ ]:
import av
import numpy as np
from IPython.display import Video, display
from PIL import Image

VIDEO_PATH = Path("lfm3_omni_moving_square.mp4")
N_FRAMES = 8
FPS = 2
SIZE = 320

container = av.open(str(VIDEO_PATH), mode="w")
stream = container.add_stream("mpeg4", rate=FPS)
stream.width = SIZE
stream.height = SIZE
stream.pix_fmt = "yuv420p"
for i in range(N_FRAMES):
    frame = np.full((SIZE, SIZE, 3), (20, 70, 50), dtype=np.uint8)
    y = int((i / max(N_FRAMES - 1, 1)) * (SIZE - 64))
    frame[y : y + 64, 128:192] = (250, 80, 40)
    vf = av.VideoFrame.from_ndarray(frame, format="rgb24")
    for packet in stream.encode(vf):
        container.mux(packet)
for packet in stream.encode():
    container.mux(packet)
container.close()

display(Video(str(VIDEO_PATH), embed=True, width=320))
print(generate_text(
    [
        {
            "role": "user",
            "content": [
                {"type": "video", "video": str(VIDEO_PATH)},
                {"type": "text", "text": "What happens in this video?"},
            ],
        }
    ],
    videos_kwargs={"max_frames": N_FRAMES, "fps": float(FPS), "min_frames": 4},
))


## Optional GGUF

[`LiquidAI/LFM3-Omni-2B-GGUF`](https://huggingface.co/LiquidAI/LFM3-Omni-2B-GGUF) has `Q4_K_M` (~1.1 GB) plus a ~0.5–0.9 GB mmproj. Omni llama.cpp is still catching up; Transformers is the primary T4 path.


## Caveats

- **Licence:** [LFM Open License v1.0](https://huggingface.co/LiquidAI/LFM3-Omni-2B/blob/main/LICENSE) (`license: other`).
- **T4 caps:** 8-frame / 2 fps video, text-only generate, float16. Speech out and video+soundtrack are card features; skip them on 15 GB.
- **Install:** `transformers>=5.20.2` + `trust_remote_code`. Native LFM3-Omni is still [PR #50036](https://github.com/huggingface/transformers/pull/50036).
- **HF token:** public weights. Cancel Colab `HF_TOKEN` prompts.
- **Not executed here:** no GPU in the agent VM. Full T4 `generate()` was not run.
- **Do not merge from this agent.**
